# Python API reference

The [quickstart](quickstart.ipynb) introduces the workflow; this page is a
lookup reference for the Python call and report methods. The profiler
accepts an in-memory Polars DataFrame, a Polars LazyFrame, or a path to a
Parquet, CSV, or IPC file. The Rust API and JSON schema expose
the same report concepts.

## `onceover.profile`

```python
profile(
    df, budget="standard", deadline_seconds=None, *,
    sample_rows=None, max_predictors=None, max_pairs=None,
    detectors=None, max_findings=100, max_rules=20,
    hints=None, target=None, timing=False,
    source_row_cap=None, exact=False, sample_seed=0,
)
```

`budget` selects `quick`, `standard`, or `deep`. `deadline_seconds` is a
finite, nonnegative soft limit checked between passes, and at work
quanta inside the dependence screen, the hotspots scan, the
discontinuities pass, the surprises scan, and the duplicate-encodings
scan. Passes not yet started are skipped; those five searches can also
stop mid-pass, and their coverage entries state how much of the search
space was examined. Coverage identifies all work
skipped or stopped after the deadline.

`sample_rows` overrides the shared encoding/FD/model row limits and the
views fitting sample. `max_predictors` limits shortlisted model inputs;
`max_pairs` limits detailed numeric-pair statistics after broad screening.
These positive integer overrides affect particular searches, not every
method's independent caps. Column counts, duplicate checks, single-column
keys, and identity residual verification still use the full input.

`detectors=None` selects the defaults: 30 of the 39 registered
detectors. A list replaces that selection; `[]` returns column
summaries and coverage without detector findings. Required resource
passes can also execute. Eight detectors are opt-in and run only
when named: `views`, `dates`, `pivot`, `period_associations`,
`drift`, `granularity`, `changeforest`, and `slice_shifts`; a default run records
each as skipped in coverage.
`expectations` and `group_contrast` are in the default set but
engage only on declared hints. Discover IDs and each detector's activation with
`onceover.available_detectors()` or the [capability guide](capabilities.ipynb).

`max_findings` is a positive report-wide limit; `max_rules` limits selected
conditional rules and can be zero. Defaults are 100 findings and 20 rules.
Other per-method and per-target limits still apply.

`target` designates a numeric response for `target_shapes`. With default
selection, it enables that pass in addition to the defaults. With an
explicit list, the pass runs only when included. This argument does not
retarget every other detector or fit a deployable prediction model.

`timing=True` records `elapsed_ms` in coverage. Timings vary between runs;
the findings are unchanged. For repeatable report JSON, leave both timing
and the wall-clock deadline disabled.

LazyFrame and file inputs are collected before profiling. A source with
more rows than the source row cap (default 200,000; set it with
`source_row_cap`) is profiled on a deterministic stratified sample, and
the report's `source` section records the source's full row count, the
basis (`full` or `sampled`), and the sample's method, seed, and cap. On a
sampled basis every finding's evidence scope records the sample and the
source population — the sample's row count as `rows_evaluated`, the
source's as `population_rows`.
Pass `exact=True` to collect a lazy or file source whole. The source
controls apply only to LazyFrame and file inputs.

## `onceover.profile_many`

```python
profile_many(tables, *, overlap_threshold=0.9)
```

`profile` reads one table. `profile_many` reads a set of related
tables — a mapping of table name to DataFrame (LazyFrames are
collected in full, because containment is exact) — and reports the
structure *between* them: candidate foreign keys, found as inclusion
dependencies. A column pair from different tables is a candidate
when every distinct non-null value of the child column appears in a
parent column that is unique over its non-null values.

Every candidate is reported; they are ranked in two tiers:

- **Strong**: fully contained, and the column names agree after
  normalisation (case folded, underscores and spaces removed) —
  `orders.customer_id` into `customers.customer_id`.
- **Lower tier**: fully contained, names differ. Each carries a
  *chance score*: the probability that a value set of the same size
  would land inside the parent's values by accident. For integer
  columns the score is priced against the child's own span, because
  the canonical accident is one dense integer range containing
  another — every `1..k` column sits inside every larger id column,
  so such a candidate scores near 1 and the values alone carry no
  evidence either way. For other types the score is priced against
  every distinct value those columns use across the input tables.
  A real key whose names differ (a `support_rep` column referencing
  `employee_id`) sits in this tier by design: reported, with its
  score, not suppressed.

Pairs into a unique parent that fall short of full containment are
reported separately as *overlaps* when their shared distinct-value
share reaches `overlap_threshold`: the exact count of orphan rows
(non-null child rows whose value has no parent match; nulls are
never orphans), the orphan distinct values, and example orphans.

The result is a `MultiTableReport`, separate from the single-table
report — that contract is unchanged. It exposes `candidates` and
`overlaps` (lists of records, also as `candidates_frame()` /
`overlaps_frame()`), `coverage` (how many pairs were considered at
each stage, and which columns were not examined — same-table pairs,
boolean columns, and columns of unexamined types), `to_digest()`,
and `to_json()`. `profile_many` reports cross-table structure only;
it does not profile the individual tables.

## Declared hints

`hints` is a dictionary with these supported keys:

```python
hints = {
    "time": "date",                       # time-axis column name
    "order_by": ["entity", "date"],       # lexicographic row order
    "no_gaps": ["date", "sequence_id"],   # expected complete sequences
    "cadence": {"date": "monthly"},      # calendar or numeric step
    "dimensions": ["entity", "region"],  # panel grouping columns
    "measures": ["revenue"],             # values expected in panel cells
    "roles": {"row_no": "index"},        # semantic role overrides
    "hierarchies": [["unit", "team", "division"]],  # known hierarchies,
                                   # levels ordered finest to coarsest
    "exclude_columns": ["internal_note"],  # columns not to profile
    "labels": {"wk_act_usr": "weekly active users"},  # display labels
                                   # used in claim strings
    "contrast": {"column": "region", "value": "north"},
                                   # one subset of rows, profiled
                                   # against the rest of the table
}
```

Cadences are `daily`, `business_day`, `weekly`, `monthly`, or a positive
integer day/sequence step. A `roles` value names one of `identifier`,
`index`, `timestamp`, `timestamp_part`, `dimension`, `measure`, or
`text` for each column it covers. List-valued hints require lists, not bare
strings. Unusable column names and types are explained in expectations
coverage. Only supply hints that describe the actual table.

A `hierarchies` entry declares that its columns form one hierarchy,
ordered finest to coarsest: each level determines the next. The
expectations pass verifies every adjacent mapping exactly. A hierarchy
that holds is stated once, as a declared structural statement, and the
inferred statements that restate the chain group under it; a hierarchy
that does not hold is stated once as a deviation naming every broken
mapping. `exclude_columns` removes columns before profiling starts:
an excluded column gets no card and no finding can name it, and the
`column_exclusions` coverage entry states the exclusion and its source;
declared names that are not columns of the table are stated there as
ignored.

A `labels` entry gives a column a display label. Claims and
interpretations then name the column by its label, so a report can be
shown to readers who do not know the raw column names. Substitution
is scoped to each finding: a finding's prose uses a column's label
only when the finding's own subject or specification names that
column, so a column named like an ordinary word is never replaced
in prose that merely uses the word. The
machine-readable record does not change: subjects, specifications,
evidence metrics, and column cards keep the raw name, and finding ids
are derived from the canonical statement rather than the claim text,
so they are identical with and without labels. Labels are
presentation, never identity. A label that names a different column
of the table, one label declared for two columns, an empty label, or
a label for a column the table does not have is not applied: the raw
name stays in the prose, and the report's `labels` coverage entry
states each case and lists the labels that were applied.

A `contrast` entry declares one subset of the rows: the rows where the
named column equals the value, or — for a boolean column, with no
value — the rows where it is true. The `group_contrast` pass compares
that subset against the rest of the table on every other column and
reports the differences the sizes support, largest first. A group
that cannot be formed (an unknown column, an empty group, a group
covering every assigned row, or fewer than 10 rows on a side) is
stated in the pass coverage instead of raising. See
[group contrast](../methodology/group_contrast.ipynb) for the exact
bars.

The column the frame model assembles as the report's time axis is
excluded from target roles and from being profiled as a measure by the
series pass by structure, whatever role it was assigned — a bare week
integer can carry the measure role. A `roles` entry for the axis column
is an explicit declaration and wins over that structural default, and
`time` chooses which column the axis is. See
[declared expectations](../methodology/expectations.ipynb) for the exact
checks and [drift](../methodology/drift.ipynb) for the use of `time` by the
model-routing detector.

## Report data

- `report.overview`: shape, headline, type/role counts, and top claims.
- `report.columns`: column cards keyed by name.
- `report.findings`: the selected, ranked statements with their evidence.
- `report.coverage`: pass status, details, and optional timings.
- `report.groups`: findings connected through shared columns; membership
  does not assert that columns are redundant.
- `report.data`: the complete JSON-shaped dictionary, including the
  schema version and selection counts.
- `report.associations()`: every pair the association screens tested,
  as a DataFrame — effect sizes, raw p-values, Benjamini–Hochberg
  q-values, and the reporting tier (`strong`, `weak`, or `none`),
  including pairs that earned no finding.

These Python dictionaries are live objects. Use a copy if you want to
annotate them without changing the cached Python view. Native JSON and
Markdown renderings come from the original native report.

## Find and compare statements

```python
report.select(category="dependencies", type="functional", target="total")
report.select(column="region")
report.finding("a-finding-id")
report.findings_frame()
report.diff(other_report)
report.verify(df)
report.rows("a-finding-id", df)
report.to_checks()
```

`select` combines all supplied filters and returns complete findings in
report order. `column` matches any subject column; `target` matches the
subject target where one is defined. No filters returns all findings.
`finding` returns one finding or `None`.

`findings_frame` is a Polars index of the ledger: IDs, categories, types,
columns, inputs, targets, scores, claims, and evidence counts. Retrieve the
complete finding for its detailed metrics and scope.

`diff` treats this report as the base and the argument as the new report.
It matches stable IDs, then reports added/removed findings, changed claims
or scores, and coverage-status changes. Evidence-only changes with the
same claim and score count as unchanged. This structural comparison is
not the statistical drift detector. See [serialization](serialization.ipynb)
for a worked example.

`verify(df)` recounts the report's statements in a frame through
Polars — which shares no code with the detectors — and returns one row
per checked quantity with status `ok`, `mismatch`, `not_checkable` (a
sampled or model-based statement, with nothing exact to recount), or
`error`. Run it on the profiled frame to audit the report, or on newer
data to see which statements still hold. `rows(id, df)` returns the
rows behind one statement — violations of a dependency or identity,
missing rows, repeated rows, extreme values, rows breaking a pattern,
or rows matching a rule's conditions — with their positions in a
`__row` column; it raises `ValueError` for statements with no row set.
`to_checks()` turns exact statements into constraints to run on new
data: dicts holding a Polars expression by default, or a Great
Expectations suite with `fmt="great_expectations"`. Statements with
exceptions, or measured on a sample, are left out.

## Render and save

```python
report.to_json()                       # complete native JSON
report.to_json(max_chars=24_000)       # budgeted view, with omissions recorded
report.to_markdown()                  # human-readable report
report.to_digest(budget_chars=4_000)   # short index into the full report
report.to_html()                      # self-contained HTML (needs the [html] extra)
report.save("profile.json")
report.save("profile.md", fmt="markdown")
report.save("profile.html", fmt="html")
```

The HTML rendering carries the same claims and numbers as the JSON —
findings with their categories and scores, one card per column with an
Altair chart where the card supports one (a five-number strip for a
numeric column, the most frequent values otherwise), and the coverage
table — with the Vega bundle inlined once, so the file opens offline
and makes no external requests. It needs the `html` extra:
`pip install 'onceover[html]'`. The [command line](cli.ipynb) prints
the same renderings for a file.

A size-budgeted JSON view keeps high-ranked findings in full and indexes
others, then reduces column and coverage detail if needed. It raises
`ValueError` below the minimal skeleton. A digest rolls up similar
findings and states omissions. Read the full evidence before using either
as the basis for an automated action.

The versioned contract is in the [JSON reference](../appendix/report_contract.ipynb)
and the repository's [machine-readable schema](https://github.com/fcela/onceover/blob/main/docs/report-schema.json).


## A worked multi-table example

Three small tables: orders reference customers by name-matched
key, and customers carry a `support_rep` column that references
employees under a different name.


In [ ]:
import polars as pl
import onceover

customers = pl.DataFrame({
    "customer_id": [1, 2, 3],
    "support_rep": [2, 1, 2],
})
employees = pl.DataFrame({"employee_id": [1, 2, 3]})
orders = pl.DataFrame({
    "order_id": [10, 11, 12, 13],
    "customer_id": [1, 2, 2, 3],
})
report = onceover.profile_many(
    {"customers": customers, "employees": employees, "orders": orders})
print(report.to_digest())


In [ ]:
# One order points at a customer that does not exist: the pair
# leaves the candidate list and appears as an overlap with the
# orphan counted exactly.
customers = pl.DataFrame({"customer_id": list(range(1, 11))})
orders = pl.DataFrame({
    "order_id": list(range(10, 20)),
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 99],
})
report = onceover.profile_many({"customers": customers, "orders": orders})
record = next(o for o in report.overlaps if o["child_table"] == "orders")
print(record)
